# Part 2 (35 points) Image Classification using SVM


In this problem, we will use Support Vector Machines (SVMs) to build a binary image classifier. We will be solving the SVM optimization problem using a general purpose convex optimization package CVXOPT as well as using a scikit-learn library function based on a customized solver known as LIBSVM.


The dataset consists of 6, 862 images across 11 different weather classes. The dataset has been taken from kaggle ( kaggle link). For the purpose of this assignment, we have split the data into test and train, which can be accessed from the link: Assignment2 starter code. Each class has its own folder containing images of varying resolutions.

Before training the SVM, some pre-processing steps must be applied. Each image must then be resized to 100×100 pixels, followed by center cropping to ensure uniformity. Since SVMs require numerical feature vectors, each RGB image of size 100 × 100 × 3 should be flattened into a onedimensional vector of length 30, 000. As is the general practice for images, perform a min max scaling for normalization (scale range [0, 255] into [0, 1] by dividing by 255). 

Once pre-processing is complete, the SVM classifier will be trained using both approaches, and
their performance will be compared to evaluate the effectiveness of each method.

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import seaborn as sns

import nltk
import re
import string as s

from nltk.corpus import stopwords
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score, accuracy_score
from sklearn.naive_bayes import MultinomialNB 


from nltk.corpus import stopwords
from wordcloud import WordCloud

from sklearn.feature_extraction.text  import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics  import f1_score,accuracy_score
from sklearn.metrics import  confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import SGDClassifier


from nltk.stem import PorterStemmer
import string

In [1]:
%load_ext autoreload
%autoreload 2
from svm import SupportVectorMachine

In [11]:
from PIL import Image
import numpy as np
# create an empty datafram with column as image ,label and feature_vector( of size 30,000)
# read the folder and get the image and label and feature vector and append it to the dataframe
# save the dataframe as a csv file

def preprocess_image(image_path):
    # Open the image file
    img = Image.open(image_path)
    print(img.size)
    
    # Resize the image to 100x100 pixels
    img = img.resize((100, 100))
    
    # Convert the image to a numpy array
    img_array = np.array(img)
    
    # Flatten the image to a 1D array of length 30,000
    img_flattened = img_array.flatten()
    
    # Normalize the pixel values to the range [0, 1]
    img_normalized = img_flattened / 255.0
    
    return img_normalized

train_img = pd.DataFrame(columns=['image', 'label', 'feature_vector'])



for folder in os.listdir('../data/Q2/train'):
    print(folder)
    if os.path.isdir(os.path.join('../data/Q2/train', folder)):
        for file in os.listdir(os.path.join('../data/Q2/train', folder)):
            count +=1
            if file.endswith('.jpg'):
                image_path = os.path.join('../data/Q2/train', folder, file)
                feature_vector = preprocess_image(image_path)
                train_img = train_img.append({'image': image_path, 'label': folder, 'feature_vector': feature_vector}, ignore_index=True)

train_img.to_csv('../data/Q2/train.csv', index=False)
print(train_img.shape)



lightning
(400, 283)


AttributeError: 'DataFrame' object has no attribute 'append'

# A (18 points) Binary Classification:

Let d be the last 2 digits of your entry number. Take the subset of images for the classes d and (d + 1) mod 11) from the train/validation data provided to you (arranged alphabetically, i.e., dew is 0) and perform the following experiments in the context of binary classification.

# 1 (8 points) 
Download and install the CVXOPT package. Formulate the SVM dual optimization problem with a linear kernel in a form that can be solved using the CVXOPT package. The objective function should be expressed in the standard quadratic programming form αTPα + qTα + c matrix where P is an m × m matrix ( m being the number of training examples), q is an m-sized column vector and c is a constant. For your optimization problem, remember to use the constraints on αi ’s in the dual. Use the SVM formulation which can handle noise and use C = 1.0 (i.e. C in the expression 1/2wTw + C ∗Pi ξi ). You can refer this link to get a working overview of cvxopt module and it’s formulation.

1.a How many support vectors do you get in this case? What percentage of training samples
constitute the support vectors?

1.b Calculate the weight vector w and the intercept term b and classify each of the examples in the test file into one of the two labels. Report the test set accuracy.You will need to carefully think about how to represent w and b in this case

1.c Reshape the support vectors corresponding to the top-5 coefficients to get images of 100 × 100 × 3 and plot these (as images). Similarly, reshape and plot the weight vector w

# 2 (5 points) 
Again use the CVXOPT package to solve the dual SVM problem using a Gaussian kernel. Think about how the P matrix will be represented. Use C = 1.0 and γ = 0.001 (i.e. γ in K(x, z) = exp−γ∗∥x−z∥2 ) for this part.

2.(a) How many support vectors do you get in this case as compared to the linear case above? How many support vectors obtained here match with the linear case above?


2.(b) Note that you may not be able to explicitly store the weight vector (w) or the intercept term (b) in this case. Use your learned model to classify the test examples and report the test accuracy.


2.(c) Reshape the support vectors corresponding to the top- 5 coefficients to get images of 100 × 100 × 3 and plot these.


2.(d) Compare the test accuracy obtained here with part (a).

# 3 (5 points) 
Repeat parts -(a) & (b) with the scikit-learn SVM function, which is based on the LIBSVM package.

3.(a) Compare the nSV (Number of Support Vectors) obtained here with the first part for the linear kernel and the second part for the Gaussian kernel. How many of the support vectors obtained here match with the support vectors obtained in both these cases?


3.(b) Compare weight (w), bias (b) obtained here with the first part for linear kernel.


3.(c) Report the test accuracy for both linear and Gaussian kernel.


3.(d) Compare the computational cost (training time) of the CVXOPT with the sklearn implementation in both the linear and Gaussian case.

# 4 (4 points) 
SVM objective can also be optimized using the SGD algorithm. Report the training time and accuracy on the given dataset. How does the SGD solver fair against LIBLINEAR?

# B (17 points) Multi-Class Image Classification:
In this section, we will use the full subset of data provided in Question 2 to tackle a multi-class classification problem using Support Vector Machines (SVMs). For this task, we will utilize the Gaussian kernel to capture complex decision boundaries and improve classification performance.

# 5. (4 points) 
In class, we described the SVM formulation for a binary classification problem. In order to extend this to the multi-class setting, we train a model on each pair of classes to get kC2 classifiers, k being the number of classes (here, k = 11). During prediction time, we output the class which has the maximum number of votes from all the kC2 classifiers. You can read more about one-vs-one classifier setting at the following link. Using your CVXOPT solver from previous section, implement one-vs-one multi-class SVM. Use a Gaussian Kernel with C = 1.0 and γ = 0.001.

5.(a) Classify the test examples and report test set accuracy. In case of ties, choose the label
with the highest score.

# 6 (3 points) 
Now train a multi-class SVM on this dataset using the scikit-learn SVM function, which is based on the LIBSVM package. Repeat part (a) using a Gaussian kernel with γ = 0.001. Use C = 1.0 as earlier.

6.(a) Classify the test examples and report test set accuracy.

6.(b) How do the test set accuracy and the training time obtained here compare with part (a) above?

# 7. (4 points) 
Draw the confusion matrix for both of the above parts CVXOPT and LIBSVM. What do you observe? Which classes are miss-classified into which ones most often? Visualize (and report) 10 examples of misclassified objects. Do the results make sense? Comment.

# 8. (6 points) 
The validation set is typically used to estimate the optimal value of model hyperparameters, such as C in our SVM with a Gaussian kernel. This is done by randomly selecting a small subset of the training data as the validation set, training the model on the remaining training data, and then evaluating its performance on the validation set. For a more detailed introduction, you can refer to this video. You can check the correctness of your intuition by trying this test.

A more systematic approach to hyper-parameter tuning is K-fold cross-validation, which is commonly used in practice. In this technique, the training data is divided into K equal parts (folds). Each fold is used as a validation set once, while the remaining K-1 folds are used for training. This process is repeated for a range of hyper-parameter values, and the hyperparameters yielding the highest K-fold cross-validation accuracy are selected as the best. You can read more about cross-validation here 1 (see Section 1) for more details.

For this problem, we will perform 5-fold cross-validation to determine the optimal C value for the Gaussian kernel SVM. The test data should remain untouched throughout this process. We will use the scikit-learn SVM function to implement this approach.

8.(a) Fix γ as 0.001 and vary the value of C in the set {10−5, 10−3, 1, 5, 10} and compute the 5 -fold cross validation accuracy and the test accuracy for each value of C.

8.(b) Now, plot both the 5-fold cross validation accuracy as well as the test set accuracy on a graph as you vary the value of C on x-axis (you may use log scale on x-axis). What do you observe? Which value of C gives the best 5 -fold cross validation accuracy?


8.(c) Train an SVM classifier using the above found hyperparameter C (on the entire train set) and report the accuracy. Does this value of C improve accuracy from the previous modelaccuracy of the test set? Comment on your observations.